In [1]:
from autogen_agentchat.teams import RoundRobinGroupChat
from autogen_agentchat.agents import AssistantAgent 
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.conditions import MaxMessageTermination, TextMentionTermination
from autogen_agentchat.ui import Console

ModuleNotFoundError: No module named 'autogen_agentchat'

In [5]:
model = OpenAIChatCompletionClient(model="gpt-4o-mini")

clarity_agent = AssistantAgent(
    "ClarityAgent",
    model_client = model, 
    system_message= """
        You are an expert editor focused on clarity and simplicity. 
        Your job is to eliminate ambiguity, redundancy, and make every sentence crisp and clear. 
        Don't worry about persuasion or tone — just make the message easy to read and understand.
    """
)

tone_agnet= AssistantAgent(
    "ToneAgent",
    model_client = model, 
    system_message= """
        You are a communication coach focused on emotional tone and professionalism. 
        Your job is to make the email sound warm, confident, and human — while staying professional 
        and appropriate for the audience. Improve the emotional resonance, polish the phrasing, 
        and adjust any words that may come off as stiff, cold, or overly casual.
    """
)

persuasion_agent = AssistantAgent(
    "PersuasionAgent",
    model_client = model, 
    system_message= """
        You are a persuasion expert trained in marketing, behavioral psychology, 
        and copywriting. Your job is to enhance the email's persuasive power: improve call to action, structure arguments, and emphasize benefits. Remove weak or passive language.
    """
)

synthesizer_agent = AssistantAgent(
    "SynthesizerAgent",
    model_client = model, 
    system_message= """
        You are an advanced email-writing specialist. Your role is to read all 
        prior agent responses and revisions, and then **synthesize the best ideas** into a unified, 
        polished draft of the email. Focus on: Integrating clarity, tone, and persuasion improvements; 
        Ensuring coherence, fluency, and a natural voice; Creating a version that feels professional, 
        effective, and readable.
    """
)

critic_agent = AssistantAgent(
    "CriticAgent",
    model_client = model, 
    system_message= """
    You are an email quality evaluator. Your job is to perform a final review 
            of the synthesized email and determine if it meets professional standards. Review the email for: 
            Clarity and flow, appropriate professional tone, effective call-to-action, and overall coherence.
            Be constructive but decisive. If the email has major flaws (unclear message, unprofessional tone, 
            or missing key elements), provide ONE specific improvement suggestion. 
            If the email meets professional standards and communicates effectively, respond with 'The email meets professional standards.' followed by `TERMINATE` on a new line. 
            You should only approve emails that are perfect enough for professional use, dont settle."
    """
)

In [8]:
text_termination = TextMentionTermination("TERMINATE")
max_message_termination = MaxMessageTermination(max_messages=10)
termination_condition = text_termination | max_message_termination

In [9]:
team = RoundRobinGroupChat(
    participants=[
        clarity_agent,
        tone_agnet,
        persuasion_agent,
        synthesizer_agent,
        critic_agent,
    ],
    termination_condition=termination_condition,
)

await Console(
    team.run_stream(
        task="Hi! im hungry, buy me lunch and invest in my business. thanks"
    )    
)

---------- TextMessage (user) ----------
Hi! im hungry, buy me lunch and invest in my business. thanks
---------- TextMessage (ClarityAgent) ----------
Hello! I'm hungry. Please buy me lunch and invest in my business. Thank you.
---------- TextMessage (ToneAgent) ----------
Subject: Lunch and Business Opportunity

Hi there!

I hope this message finds you well. I wanted to reach out and see if you would be open to having lunch together. It would be a great opportunity to discuss some exciting ideas I have for my business, and I would appreciate your insights and support.

Thank you for considering this. Looking forward to hearing from you!

Best regards,  
[Your Name]
---------- TextMessage (PersuasionAgent) ----------
Subject: Let’s Connect Over Lunch and Explore an Exciting Business Opportunity!

Hi [Recipient's Name],

I hope you’re having a fantastic day! I’d like to invite you to lunch – not just to satisfy our appetites, but to ignite a conversation about an exciting opportunity I

TaskResult(messages=[TextMessage(id='feadc7d1-6b46-4ce5-8b7b-eec037d0953d', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 6, 19, 4, 51, 13, 224032, tzinfo=datetime.timezone.utc), content='Hi! im hungry, buy me lunch and invest in my business. thanks', type='TextMessage'), TextMessage(id='d74c7426-2059-413b-b943-a8eee2a14e4b', source='ClarityAgent', models_usage=RequestUsage(prompt_tokens=79, completion_tokens=18), metadata={}, created_at=datetime.datetime(2026, 6, 19, 4, 51, 16, 437970, tzinfo=datetime.timezone.utc), content="Hello! I'm hungry. Please buy me lunch and invest in my business. Thank you.", type='TextMessage'), TextMessage(id='2a89163b-7d11-4ebf-b580-2c78c7754dd6', source='ToneAgent', models_usage=RequestUsage(prompt_tokens=125, completion_tokens=82), metadata={}, created_at=datetime.datetime(2026, 6, 19, 4, 51, 18, 854116, tzinfo=datetime.timezone.utc), content='Subject: Lunch and Business Opportunity\n\nHi there!\n\nI hope this message